In [1]:
# Codeblock 1
import torch
import torch.nn as nn

NUM_CLASSES = 80
NUM_PRIORS = 3

In [4]:
# Codeblock 2
class CBS(nn.Module):
    def __init__(self, 
                 in_channels,     #(1)
                 out_channels, 
                 kernel_size=1, 
                 stride=1):       #(2)
        super().__init__()
        
        if kernel_size == 1:      #(3)
            padding = 0
        elif kernel_size == 3:    #(4)
            padding = 1
        elif kernel_size == 6:    #(5)
            padding = 2
        
        self.conv = nn.Conv2d(in_channels=in_channels,
                              out_channels=out_channels, 
                              kernel_size=kernel_size, 
                              stride=stride,
                              padding=padding,
                              bias=False)
        self.bn = nn.BatchNorm2d(num_features=out_channels)
        self.act = nn.SiLU()
        
    def forward(self, x):
        #print(f'original\t: {x.size()}')
        
        x = self.conv(x)          #(6)
        #print(f'after conv\t: {x.size()}')
        
        x = self.bn(x)            #(7)
        #print(f'after bn\t: {x.size()}')
        
        x = self.act(x)           #(8)
        #print(f'after act\t: {x.size()}')
        
        return x

In [3]:
# Codeblock 3
cbs = CBS(in_channels=3,
          out_channels=64,
          kernel_size=6, 
          stride=2)

x = torch.randn(1, 3, 640, 640)    #(1)
out = cbs(x)

original	: torch.Size([1, 3, 640, 640])
after conv	: torch.Size([1, 64, 320, 320])
after bn	: torch.Size([1, 64, 320, 320])
after act	: torch.Size([1, 64, 320, 320])


In [7]:
# Codeblock 4
class Bottleneck(nn.Module):
    def __init__(self, num_channels, shortcut=True):    #(1)
        super().__init__()
        
        self.shortcut = shortcut
        
        self.conv0 = CBS(in_channels=num_channels,      #(2)
                         out_channels=num_channels)
        
        self.conv1 = CBS(in_channels=num_channels,      #(3)
                         out_channels=num_channels,
                         kernel_size=3)

    def forward(self, x):
        original = x.clone()
        #print(f'original\t: {x.size()}')
        
        x = self.conv0(x)
        #print(f'after conv0\t: {x.size()}')
        
        x = self.conv1(x)
        #print(f'after conv1\t: {x.size()}')
        
        if self.shortcut:      #(4)
            x = x + original
            #print(f'after summation\t: {x.size()}')
        
        return x

In [6]:
# Codeblock 5
bottleneck = Bottleneck(num_channels=64, shortcut=True)

x = torch.randn(1, 64, 160, 160)
out = bottleneck(x)

original	: torch.Size([1, 64, 160, 160])
after conv0	: torch.Size([1, 64, 160, 160])
after conv1	: torch.Size([1, 64, 160, 160])
after summation	: torch.Size([1, 64, 160, 160])


In [10]:
# Codeblock 6
class C3(nn.Module):
    def __init__(self, 
                 in_channels, 
                 out_channels, 
                 num_blocks,       #(1)
                 shortcut=True):
        super().__init__()
        
        self.split0 = CBS(in_channels=in_channels,       #(2)
                          out_channels=in_channels//2)
        
        self.split1 = CBS(in_channels=in_channels,       #(3)
                          out_channels=in_channels//2)
        
        self.bottlenecks = nn.ModuleList([Bottleneck(num_channels=in_channels//2) for _ in range(num_blocks)])    #(4)
        
        self.transition = CBS(in_channels=in_channels,   #(5)
                              out_channels=out_channels)

    def forward(self, x):
        #print(f'original\t\t: {x.size()}')
        
        x0 = self.split0(x)                #(6)
        #print(f'after split0 (x0)\t: {x0.size()}')
        
        x1 = self.split1(x)                #(7)
        #print(f'after split1 (x1)\t: {x1.size()}')
        
        for i, bottleneck in enumerate(self.bottlenecks):    #(8)
            x1 = bottleneck(x1)
            #print(f'x1 after bottleneck #{i}\t: {x1.size()}')
        
        x  = torch.cat([x0, x1], dim=1)    #(9)
        #print(f'after concatenate\t: {x.size()}')
        
        x  = self.transition(x)            #(10)
        #print(f'after transition\t: {x.size()}')
        
        return x

In [9]:
# Codeblock 7
c3 = C3(in_channels=128, out_channels=128, num_blocks=3)

x = torch.randn(1, 128, 160, 160)
out = c3(x)

original		: torch.Size([1, 128, 160, 160])
after split0 (x0)	: torch.Size([1, 64, 160, 160])
after split1 (x1)	: torch.Size([1, 64, 160, 160])
x1 after bottleneck #0	: torch.Size([1, 64, 160, 160])
x1 after bottleneck #1	: torch.Size([1, 64, 160, 160])
x1 after bottleneck #2	: torch.Size([1, 64, 160, 160])
after concatenate	: torch.Size([1, 128, 160, 160])
after transition	: torch.Size([1, 128, 160, 160])


In [13]:
# Codeblock 8
class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.stem = CBS(in_channels=3, 
                        out_channels=64, 
                        kernel_size=6, 
                        stride=2)
        
        
        self.downsample0 = CBS(in_channels=64, 
                               out_channels=128, 
                               kernel_size=3, 
                               stride=2)
        self.c3_0  = C3(in_channels=128, 
                        out_channels=128, 
                        num_blocks=3)
        
        
        self.downsample1 = CBS(in_channels=128, 
                               out_channels=256, 
                               kernel_size=3, 
                               stride=2)
        self.c3_1  = C3(in_channels=256, 
                        out_channels=256, 
                        num_blocks=6)
        
        
        self.downsample2 = CBS(in_channels=256, 
                               out_channels=512, 
                               kernel_size=3, 
                               stride=2)
        self.c3_2  = C3(in_channels=512, 
                        out_channels=512, 
                        num_blocks=9)
        
        
        self.downsample3 = CBS(in_channels=512, 
                               out_channels=1024, 
                               kernel_size=3, 
                               stride=2)
        self.c3_3  = C3(in_channels=1024, 
                        out_channels=1024, 
                        num_blocks=3)

    def forward(self, x):
        #print(f'original\t: {x.size()}')
        
        x2 = self.stem(x)
        #print(f'after stem\t: {x2.size()}')
        
        x4 = self.downsample0(x2)
        x4 = self.c3_0(x4)
        #print(f'after c3_0\t: {x4.size()}')
        
        x8 = self.downsample1(x4)
        x8 = self.c3_1(x8)
        #print(f'after c3_1\t: {x8.size()}')
        
        x16 = self.downsample2(x8)
        x16 = self.c3_2(x16)
        #print(f'after c3_2\t: {x16.size()}')
        
        x32 = self.downsample3(x16)
        x32 = self.c3_3(x32)
        #print(f'after c3_2\t: {x32.size()}')
        
        return x8, x16, x32

In [12]:
# Codeblock 9
backbone = Backbone()
x = torch.randn(1, 3, 640, 640)
 
out = backbone(x)

original	: torch.Size([1, 3, 640, 640])
after stem	: torch.Size([1, 64, 320, 320])
after c3_0	: torch.Size([1, 128, 160, 160])
after c3_1	: torch.Size([1, 256, 80, 80])
after c3_2	: torch.Size([1, 512, 40, 40])
after c3_2	: torch.Size([1, 1024, 20, 20])


In [16]:
# Codeblock 10
class SPPF(nn.Module):
    def __init__(self, num_channels=1024):
        super().__init__()
        
        self.conv0 = CBS(in_channels=num_channels,                       #(1)
                         out_channels=num_channels//2)
        
        self.maxpool = nn.MaxPool2d(kernel_size=5, stride=1, padding=2)  #(2)
        
        self.conv1 = CBS(in_channels=num_channels*2,                     #(3)
                         out_channels=num_channels)

    def forward(self, x):
        #print(f'original\t\t: {x.size()}')
        
        x = self.conv0(x)                #(4)
        #print(f'after conv0\t\t: {x.size()}')
        
        pooled0 = self.maxpool(x)        #(5)
        #print(f'after maxpool0\t\t: {pooled0.size()}')
        
        pooled1 = self.maxpool(pooled0)  #(6)
        #print(f'after maxpool1\t\t: {pooled1.size()}')
        
        pooled2 = self.maxpool(pooled1)  #(7)
        #print(f'after maxpool2\t\t: {pooled2.size()}')
        
        x = torch.cat([x, pooled0, pooled1, pooled2], dim=1)    #(8)
        #print(f'after concat\t\t: {x.size()}')
        
        x = self.conv1(x)                #(9)
        #print(f'after conv1\t\t: {x.size()}')
        
        return x                         #(10)

In [15]:
# Codeblock 11
sppf = SPPF()
x32 = torch.randn(1, 1024, 20, 20)

out = sppf(x32)

original		: torch.Size([1, 1024, 20, 20])
after conv0		: torch.Size([1, 512, 20, 20])
after maxpool0		: torch.Size([1, 512, 20, 20])
after maxpool1		: torch.Size([1, 512, 20, 20])
after maxpool2		: torch.Size([1, 512, 20, 20])
after concat		: torch.Size([1, 2048, 20, 20])
after conv1		: torch.Size([1, 1024, 20, 20])


In [19]:
# Codeblock 12
class PAN(nn.Module):
    def __init__(self):
        super().__init__()
        
        ## Layers for top-down flow.
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')
        
        self.conv_x32_to_x16 = CBS(1024, 512, 1, 1)
        self.conv_combined_x16_td = C3(1024, 512, num_blocks=3, shortcut=False)
        
        self.conv_x16_to_x8 = CBS(512, 256, 1, 1)
        self.conv_combined_x8_td = C3(512, 256, num_blocks=3, shortcut=False)
        
        
        ## Layers for bottom-up flow.
        self.conv_x8_to_x16 = CBS(256, 256, 3, 2)
        self.conv_combined_x16_bu = C3(512, 512, num_blocks=3, shortcut=False)
        
        self.conv_x16_to_x32 = CBS(512, 512, 3, 2)
        self.conv_combined_x32_bu = C3(1024, 1024, num_blocks=3, shortcut=False)

        
    def forward(self, x8, x16, x32):
        
        ##### Top-down flow.
        
        x32_reduced = self.conv_x32_to_x16(x32)
        #print(f'x32_reduced\t\t: {x32_reduced.size()}')
        
        x32_upsampled = self.upsample(x32_reduced)
        #print(f'x32_upsampled\t\t: {x32_upsampled.size()}')
        
        x16_concatenated_td = torch.cat([x32_upsampled, x16], dim=1)
        #print(f'x16_concatenated_td\t: {x16_concatenated_td.size()}')
        
        x16_combined_td = self.conv_combined_x16_td(x16_concatenated_td)
        #print(f'x16_combined_td\t\t: {x16_combined_td.size()}\n')

        x16_reduced = self.conv_x16_to_x8(x16_combined_td)
        #print(f'x16_reduced\t\t: {x16_reduced.size()}')
        
        x16_upsampled = self.upsample(x16_reduced)
        #print(f'x16_upsampled\t\t: {x16_upsampled.size()}')
        
        x8_concatenated_td = torch.cat([x16_upsampled, x8], dim=1)
        #print(f'x8_concatenated_td\t: {x8_concatenated_td.size()}')
        
        x8_combined_td = self.conv_combined_x8_td(x8_concatenated_td) 
        #print(f'x8_combined_td\t\t: {x8_combined_td.size()}\n')
        
        
        
        x8_out = x8_combined_td
        #print(f'x8_out\t\t\t: {x8_out.size()}\n')
        
        
        ##### Bottom-up flow.
        
        x8_downsampled = self.conv_x8_to_x16(x8_combined_td)
        #print(f'x8_downsampled\t\t: {x8_downsampled.size()}')
        
        x16_concatenated_bu = torch.cat([x8_downsampled, x16_reduced], dim=1)
        #print(f'x16_concatenated_bu\t: {x16_concatenated_bu.size()}')
        
        x16_combined_bu = self.conv_combined_x16_bu(x16_concatenated_bu) 
        #print(f'x16_combined_bu\t\t: {x16_combined_bu.size()}\n')
        
        
        
        x16_out = x16_combined_bu
        #print(f'x16_out\t\t\t: {x16_out.size()}\n')
        
        
        
        x16_downsampled = self.conv_x16_to_x32(x16_combined_bu)
        #print(f'x16_downsampled\t\t: {x16_downsampled.size()}')
        
        x32_concatenated_bu = torch.cat([x16_downsampled, x32_reduced], dim=1)
        #print(f'x32_concatenated_bu\t: {x32_concatenated_bu.size()}')
        
        x32_combined_bu = self.conv_combined_x32_bu(x32_concatenated_bu)
        #print(f'x32_combined_bu\t\t: {x32_combined_bu.size()}\n')
        
        
        
        x32_out = x32_combined_bu
        #print(f'x32_out\t\t\t: {x32_out.size()}')
        
        return x8_out, x16_out, x32_out

In [18]:
# Codeblock 13
pan = PAN()
x8  = torch.randn(1, 256, 80, 80)
x16 = torch.randn(1, 512, 40, 40)
x32 = torch.randn(1, 1024, 20, 20)

out = pan(x8, x16, x32)

x32_reduced		: torch.Size([1, 512, 20, 20])
x32_upsampled		: torch.Size([1, 512, 40, 40])
x16_concatenated_td	: torch.Size([1, 1024, 40, 40])
x16_combined_td		: torch.Size([1, 512, 40, 40])

x16_reduced		: torch.Size([1, 256, 40, 40])
x16_upsampled		: torch.Size([1, 256, 80, 80])
x8_concatenated_td	: torch.Size([1, 512, 80, 80])
x8_combined_td		: torch.Size([1, 256, 80, 80])

x8_out			: torch.Size([1, 256, 80, 80])

x8_downsampled		: torch.Size([1, 256, 40, 40])
x16_concatenated_bu	: torch.Size([1, 512, 40, 40])
x16_combined_bu		: torch.Size([1, 512, 40, 40])

x16_out			: torch.Size([1, 512, 40, 40])

x16_downsampled		: torch.Size([1, 512, 20, 20])
x32_concatenated_bu	: torch.Size([1, 1024, 20, 20])
x32_combined_bu		: torch.Size([1, 1024, 20, 20])

x32_out			: torch.Size([1, 1024, 20, 20])


In [22]:
# Codeblock 14
class Head(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=num_channels, 
                              out_channels=NUM_PRIORS*(5+NUM_CLASSES), 
                              kernel_size=1)

    def forward(self, x):
        #print(f'original\t : {x.size()}')
        
        x = self.conv(x)
        #print(f'final prediction : {x.size()}')
        
        return x

In [21]:
# Codeblock 15
head_x8 = Head(num_channels=256)
x8 = torch.randn(1, 256, 80, 80)

out = head_x8(x8)

original	 : torch.Size([1, 256, 80, 80])
final prediction : torch.Size([1, 255, 80, 80])


In [23]:
# Codeblock 16
class YOLOv5(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.backbone = Backbone()              #(1)
        self.sppf = SPPF()                      #(2)

        self.pan = PAN()                        #(3)
        
        self.head_s = Head(num_channels=256)    #(4)
        self.head_m = Head(num_channels=512)
        self.head_l = Head(num_channels=1024)   #(5)

    def forward(self, x):
        print(f'x original\t\t: {x.size()}\n')
        
        x8, x16, x32 = self.backbone(x)         #(6)
        print(f'x8 after backbone\t: {x8.size()}')
        print(f'x16 after backbone\t: {x16.size()}')
        print(f'x32 after backbone\t: {x32.size()}\n')
        
        x32 = self.sppf(x32)                    #(7)
        print(f'x32 after sppf\t\t: {x32.size()}\n')
        
        x8, x16, x32 = self.pan(x8, x16, x32)   #(8)
        print(f'x8 after pan\t\t: {x8.size()}')
        print(f'x16 after pan\t\t: {x16.size()}')
        print(f'x32 after pan\t\t: {x32.size()}\n')
        
        x8_pred  = self.head_s(x8)              #(9)
        x16_pred = self.head_m(x16)
        x32_pred = self.head_l(x32)             #(10)
        print(f'x8 after head_s\t\t: {x8_pred.size()}')
        print(f'x16 after head_m\t: {x16_pred.size()}')
        print(f'x32 after head_l\t: {x32_pred.size()}')
        
        return x8_pred, x16_pred, x32_pred

In [24]:
# Codeblock 17
yolov5 = YOLOv5()

x = torch.randn(1, 3, 640, 640)
out = yolov5(x)

x original		: torch.Size([1, 3, 640, 640])

x8 after backbone	: torch.Size([1, 256, 80, 80])
x16 after backbone	: torch.Size([1, 512, 40, 40])
x32 after backbone	: torch.Size([1, 1024, 20, 20])

x32 after sppf		: torch.Size([1, 1024, 20, 20])

x8 after pan		: torch.Size([1, 256, 80, 80])
x16 after pan		: torch.Size([1, 512, 40, 40])
x32 after pan		: torch.Size([1, 1024, 20, 20])

x8 after head_s		: torch.Size([1, 255, 80, 80])
x16 after head_m	: torch.Size([1, 255, 40, 40])
x32 after head_l	: torch.Size([1, 255, 20, 20])
